# MNIST ONNX Model Creation

This notebook documents the conversion of the existing trained MNIST model into the ONNX format used by the Unity implementations.

The trained model and its FPGA/ZyNet preparation are documented separately. This notebook begins with the saved model file:

```text
WeigntsAndBiases.txt
```

and produces:

```text
mnist_model.onnx
```

The ONNX model was then used by Unity Inference Engine / Sentis for the PC and Meta Quest 3S implementations.

The notebook also verifies the ONNX graph, compares its predictions with the original Python model, and generates the binary MNIST test files used by Unity.

## Files used

Place the following files in the same directory as this notebook:

```text
WeigntsAndBiases.txt
network2.py
mnist_loader.py
mnist.pkl.gz

export_to_onnx.py
verify_python_accuracy.py
export_all_test.py
export_one_test.py
```

The workflow creates or verifies these artifacts:

```text
mnist_model.onnx
mnist_test_images.bin
mnist_test_labels.bin
```

The spelling of `WeigntsAndBiases.txt` is kept because it is the filename used by the project.

## 1. Inspect the saved model

The saved model contains the network architecture, weights, biases, and cost-function name. The ONNX exporter expects the architecture used in the Unity experiments:

```text
784 -> 30 -> 20 -> 10
```

with sigmoid activation after each fully connected layer.

In [ ]:
import json
import numpy as np

MODEL_FILE = "WeigntsAndBiases.txt"

with open(MODEL_FILE, "r") as f:
    model_data = json.load(f)

print("Network sizes:", model_data["sizes"])
print("Weight matrices:", len(model_data["weights"]))
print("Bias vectors:", len(model_data["biases"]))
print("Cost function:", model_data["cost"])

if model_data["sizes"] != [784, 30, 20, 10]:
    raise ValueError(f"Unexpected network architecture: {model_data['sizes']}")

### Parameter dimensions

The expected weight and bias shapes are:

| Layer | Weight shape | Bias shape |
|---|---:|---:|
| 784 → 30 | 30 × 784 | 30 × 1 |
| 30 → 20 | 20 × 30 | 20 × 1 |
| 20 → 10 | 10 × 20 | 10 × 1 |

The network contains 24,350 trainable parameters.

In [ ]:
weights = [np.asarray(w) for w in model_data["weights"]]
biases = [np.asarray(b) for b in model_data["biases"]]

for i, (w, b) in enumerate(zip(weights, biases), start=1):
    print(f"Layer {i}: weights {w.shape}, biases {b.shape}")

parameter_count = sum(w.size + b.size for w, b in zip(weights, biases))
print("Trainable parameters:", parameter_count)

assert parameter_count == 24350

## 2. Verify the Python model before export

Before ONNX conversion, the saved model can be loaded with `network2.py` and evaluated against the complete MNIST test set.

The model used in the study classified:

```text
9,663 / 10,000 = 96.63%
```

This provides the software reference result for checking the ONNX conversion.

In [ ]:
import mnist_loader
import network2

_, _, test_data = mnist_loader.load_data_wrapper()
test_data = list(test_data)

python_net = network2.load(MODEL_FILE)

correct = python_net.accuracy(test_data)
total = len(test_data)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total * 100:.2f}%")

## 3. Export the model to ONNX

The Python implementation computes each layer as:

```text
sigmoid(Wx + b)
```

The ONNX graph represents the same computation with a `Gemm` node followed by a `Sigmoid` node for each layer.

The saved Python weights use the shape:

```text
[output_neurons, input_neurons]
```

For the ONNX `Gemm` configuration used here, each weight matrix is transposed to:

```text
[input_neurons, output_neurons]
```

before being stored as an initializer.

The exporter uses:

- input tensor: `[1, 784]`
- output tensor: `[1, 10]`
- parameter type: `float32`
- ONNX opset: 13
- ONNX IR version: 8

In [ ]:
import onnx
from onnx import helper, TensorProto, numpy_helper

OUTPUT_FILE = "mnist_model.onnx"

weights32 = [np.array(w, dtype=np.float32) for w in model_data["weights"]]
biases32 = [np.array(b, dtype=np.float32).reshape(-1) for b in model_data["biases"]]

input_info = helper.make_tensor_value_info(
    "input",
    TensorProto.FLOAT,
    [1, 784]
)

output_info = helper.make_tensor_value_info(
    "output",
    TensorProto.FLOAT,
    [1, 10]
)

nodes = []
initializers = []
previous_output = "input"

for i, (w, b) in enumerate(zip(weights32, biases32), start=1):
    weight_name = f"W{i}"
    bias_name = f"B{i}"
    gemm_output = f"gemm{i}"
    sigmoid_output = "output" if i == len(weights32) else f"sigmoid{i}"

    initializers.append(
        numpy_helper.from_array(w.T.astype(np.float32), name=weight_name)
    )
    initializers.append(
        numpy_helper.from_array(b.astype(np.float32), name=bias_name)
    )

    nodes.append(
        helper.make_node(
            "Gemm",
            inputs=[previous_output, weight_name, bias_name],
            outputs=[gemm_output],
            name=f"Dense{i}"
        )
    )

    nodes.append(
        helper.make_node(
            "Sigmoid",
            inputs=[gemm_output],
            outputs=[sigmoid_output],
            name=f"Sigmoid{i}"
        )
    )

    previous_output = sigmoid_output

graph = helper.make_graph(
    nodes,
    "MNIST_Feedforward_Network",
    [input_info],
    [output_info],
    initializer=initializers
)

onnx_model = helper.make_model(
    graph,
    producer_name="MNIST_Unity_Exporter",
    opset_imports=[helper.make_opsetid("", 13)]
)

onnx_model.ir_version = 8

onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, OUTPUT_FILE)

print(f"Successfully created {OUTPUT_FILE}")

## 4. Inspect the ONNX graph

The expected graph contains six computational nodes:

```text
Gemm -> Sigmoid -> Gemm -> Sigmoid -> Gemm -> Sigmoid
```

The final sigmoid output contains 10 activation values. The predicted digit is the index of the largest output value.

In [ ]:
model = onnx.load(OUTPUT_FILE)
onnx.checker.check_model(model)

print("Producer:", model.producer_name)
print("IR version:", model.ir_version)
print("Opset:", model.opset_import[0].version)

print("\nNodes:")
for node in model.graph.node:
    print(f"{node.name}: {node.op_type}")

print("\nInput:")
for value in model.graph.input:
    dims = [d.dim_value for d in value.type.tensor_type.shape.dim]
    print(value.name, dims)

print("\nOutput:")
for value in model.graph.output:
    dims = [d.dim_value for d in value.type.tensor_type.shape.dim]
    print(value.name, dims)

## 5. Check the ONNX artifact size

The ONNX file used by the Unity experiments occupied 97,953 bytes.

This check confirms whether the recreated artifact matches that recorded size.

In [ ]:
import os

onnx_size = os.path.getsize(OUTPUT_FILE)

print(f"ONNX size: {onnx_size:,} bytes")

if onnx_size == 97953:
    print("File size matches the model used in the Unity experiments.")
else:
    print("File size differs from the recorded 97,953-byte artifact.")
    print("Review library/version differences before replacing the project copy.")

## 6. Compare Python and ONNX inference

Structural validation confirms that the ONNX graph is legal, but it does not by itself confirm numerical equivalence.

The next check sends the same MNIST image through:

1. the Python `network2` model, and
2. the ONNX model.

The output activations, predicted class, and maximum absolute difference are then compared.

This verification is useful for repository reproducibility even if it was not part of the original conversion script.

In [ ]:
try:
    import onnxruntime as ort
except ImportError as e:
    raise ImportError(
        "onnxruntime is required for inference validation. "
        "Install it with: pip install onnxruntime"
    ) from e

session = ort.InferenceSession(
    OUTPUT_FILE,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

x, expected_label = test_data[0]

python_output = python_net.feedforward(x).reshape(-1)
onnx_input = np.asarray(x, dtype=np.float32).reshape(1, 784)
onnx_output = session.run([output_name], {input_name: onnx_input})[0].reshape(-1)

print("Expected label:", expected_label)
print("Python prediction:", int(np.argmax(python_output)))
print("ONNX prediction:", int(np.argmax(onnx_output)))
print("Maximum absolute output difference:",
      float(np.max(np.abs(python_output - onnx_output))))

## 7. Validate ONNX accuracy on the full MNIST test set

The Unity CPU and GPU implementations produced the same 96.63% controlled accuracy as the Python model.

The following validation runs the ONNX model over all 10,000 MNIST test images and checks whether it reproduces that result.

In [ ]:
onnx_correct = 0
prediction_mismatches = 0

for x, label in test_data:
    x32 = np.asarray(x, dtype=np.float32).reshape(1, 784)

    py_prediction = int(np.argmax(python_net.feedforward(x)))
    onnx_scores = session.run([output_name], {input_name: x32})[0]
    onnx_prediction = int(np.argmax(onnx_scores))

    onnx_correct += int(onnx_prediction == label)
    prediction_mismatches += int(onnx_prediction != py_prediction)

print(f"ONNX correct: {onnx_correct} / {len(test_data)}")
print(f"ONNX accuracy: {onnx_correct / len(test_data) * 100:.2f}%")
print(f"Python/ONNX prediction mismatches: {prediction_mismatches}")

## 8. Export the MNIST test data used by Unity

The Unity controlled benchmarks used binary files containing the complete MNIST test set:

```text
mnist_test_images.bin
mnist_test_labels.bin
```

The image file stores the 10,000 test images as `float32` values. Each image contains 784 values.

The label file stores the corresponding labels as `uint8` values.

These files can be recreated from `mnist.pkl.gz`; they do not require retraining or model conversion.

In [ ]:
import gzip
import pickle

MNIST_FILE = "mnist.pkl.gz"
IMAGE_OUTPUT = "mnist_test_images.bin"
LABEL_OUTPUT = "mnist_test_labels.bin"

with gzip.open(MNIST_FILE, "rb") as f:
    training_data_raw, validation_data_raw, test_data_raw = pickle.load(
        f, encoding="latin1"
    )

images, labels = test_data_raw

images = np.asarray(images, dtype=np.float32)
labels = np.asarray(labels, dtype=np.uint8)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

images.tofile(IMAGE_OUTPUT)
labels.tofile(LABEL_OUTPUT)

print("Created:", IMAGE_OUTPUT)
print("Created:", LABEL_OUTPUT)
print("Image bytes:", os.path.getsize(IMAGE_OUTPUT))
print("Label bytes:", os.path.getsize(LABEL_OUTPUT))

## 9. Optional single-image export

`export_one_test.py` was also used to create a human-readable file for one MNIST test image.

The file contains:

1. the expected digit on the first line;
2. 784 comma-separated pixel values on the second line.

This is useful for checking individual inputs independently of the complete binary dataset.

In [ ]:
SINGLE_IMAGE_OUTPUT = "mnist_test_0000.txt"

images, labels = test_data_raw
image = images[0]
label = int(labels[0])

with open(SINGLE_IMAGE_OUTPUT, "w") as f:
    f.write(str(label) + "\n")
    f.write(",".join(str(float(pixel)) for pixel in image))

print("Created:", SINGLE_IMAGE_OUTPUT)
print("Expected digit:", label)
print("Pixel count:", len(image))

## 10. Final Unity model artifacts

The files produced or verified by this workflow are:

| File | Purpose |
|---|---|
| `mnist_model.onnx` | ONNX model loaded by Unity Inference Engine / Sentis |
| `mnist_test_images.bin` | 10,000 MNIST test images used for controlled Unity testing |
| `mnist_test_labels.bin` | Labels corresponding to the 10,000 test images |
| `mnist_test_0000.txt` | Optional human-readable single-image test input |

The Unity experiments used the ONNX representation of the same trained 784–30–20–10 MNIST network. The model uses sigmoid activations and contains 24,350 trainable parameters.

The ONNX artifact does not include camera preprocessing. PC and Quest camera acquisition, ROI mapping, grayscale conversion, thresholding, and application-level benchmarking are documented with the platform implementation code rather than in this model-conversion notebook.

## Reproducibility notes

- The ONNX model is created from `WeigntsAndBiases.txt`; this notebook does not retrain the MNIST network.
- The exporter converts model parameters to `float32` for ONNX.
- Each Python weight matrix is transposed before use by the corresponding ONNX `Gemm` node.
- ONNX opset 13 and IR version 8 were used by the project exporter.
- `onnx.checker.check_model()` verifies graph validity.
- ONNX Runtime can be used to confirm that predictions match the Python implementation.
- The full MNIST test-set accuracy should reproduce the 96.63% software reference result.
- Binary Unity test files can be regenerated directly from `mnist.pkl.gz`.
- Library versions can affect the serialized ONNX file even when the represented computation is unchanged. If reproducing the exact file byte-for-byte is required, record the Python, NumPy, ONNX, and ONNX Runtime versions used.